# ODSC 2026 — Notebook 2: OpenClaw

The same compliance monitoring logic, now running in a production agent framework. You chat with it in plain English — it decides which tools to call, which pages to check, and when to alert you.

| Part | What you build |
| --- | --- |
| **Part 1** | Start OpenClaw, build `AGENTS.md`, run your first check |
| **Part 2** | Telegram notifications via OpenClaw channels |
| **Part 3** | Observability with Opik |

**Prerequisites:** Completed `custom_claw.ipynb`, Docker still running.

In [ ]:
import subprocess
import os
import shutil
from dotenv import load_dotenv

load_dotenv(override=True)

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
assert OPENAI_API_KEY and not OPENAI_API_KEY.startswith("your-"), "Set OPENAI_API_KEY in .env"

docker_ok = subprocess.run("docker info".split(), capture_output=True).returncode == 0
assert docker_ok, "Docker is not running. Start Docker Desktop and try again."

# Auto-detect Docker sudo/compose variant
SUDO = ""
if subprocess.run("docker info".split(), capture_output=True).returncode != 0:
    if subprocess.run("sudo docker info".split(), capture_output=True).returncode == 0:
        SUDO = "sudo "

DOCKER_COMPOSE = (
    f"{SUDO}docker compose"
    if subprocess.run(f"{SUDO}docker compose version".split(), capture_output=True).returncode == 0
    else f"{SUDO}docker-compose"
)
OPENCLAW_COMPOSE = f"{DOCKER_COMPOSE} -f openclaw-agent/docker-compose.yml"

shutil.copy(".env", "openclaw-agent/.env")

print("✅ OpenAI key set")
print("✅ Docker running")
print(f"Using: {DOCKER_COMPOSE}")
print("\nReady to go.")

---
## Part 1: Start OpenClaw

---
### Why OpenClaw?

Custom Claw works, but it's a single Python script with no scheduling, no dashboard, no memory, and no multi-channel support. 

[OpenClaw](https://github.com/openclaw/openclaw) is a production agent framework that wraps the same logic with:
- Persistent memory and conversation context
- Web dashboard for monitoring
- Built-in scheduling
- Telegram, Slack, Discord, WhatsApp out of the box
- 50+ built-in skills
- Skills defined as Markdown — no Python required

The `check-compliance-updates` skill in `openclaw-agent/skills/` is the OpenClaw equivalent of `monitor.py`.

### Building `AGENTS.md` — Your Agent's Domain Knowledge

With Custom Claw, domain knowledge lives in the code: hardcoded URLs, hardcoded logic, hardcoded subjects.

With an LLM-powered agent, you externalize that knowledge into `AGENTS.md` — a plain text file the agent reads as context. It tells the agent:
- **What it does** — its purpose and scope
- **Domain knowledge** — what it needs to know about teacher licensure, Praxis exams, key sources
- **Skills available** — what tools it can call
- **Behavior guidelines** — how to respond, what to cite, when to search

Open `AGENTS.md` in the project root. That file is what gives OpenClaw its expertise.

> **The pattern:** Any time you build an LLM agent, start here. Define the domain, the tools, and the behavioral guardrails *before* you write a line of code. The agent reasons from this context — this is where the intelligence lives.

In [ ]:
import shutil
import time

OPENCLAW_COMPOSE = f"{DOCKER_COMPOSE} -f openclaw-agent/docker-compose.yml"

# Sync .env
shutil.copy(".env", "openclaw-agent/.env")

print("Starting OpenClaw gateway...")
!{OPENCLAW_COMPOSE} up -d openclaw-gateway

for i in range(15):
    health = subprocess.run(
        f"{OPENCLAW_COMPOSE} exec openclaw-gateway curl -sf http://localhost:18789/healthz".split(),
        capture_output=True
    )
    if health.returncode == 0:
        print("✅ Gateway healthy!")
        break
    time.sleep(3)
    print(f"   Waiting... ({i+1})")

In [ ]:
# Configure the agent
!{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs config set agents.defaults.model openai/gpt-4o
!{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs config set gateway.bind lan
print("\n✅ Agent configured.")

In [ ]:
# Ask the agent to run a compliance check
!{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs agent -m "Check for any teacher licensure compliance updates in Ohio and Texas."

---
## Part 2: Telegram Notifications

Add `OPENCLAW_TELEGRAM_BOT_TOKEN` to your `.env`, then run this cell to pair your bot with OpenClaw.

In [ ]:
# Connect Telegram to OpenClaw (get pairing code)
print("Adding Telegram channel to OpenClaw...")

TELEGRAM_BOT_TOKEN = os.getenv("OPENCLAW_TELEGRAM_BOT_TOKEN", "")
assert TELEGRAM_BOT_TOKEN, "Set OPENCLAW_TELEGRAM_BOT_TOKEN in .env first"

!{OPENCLAW_COMPOSE} exec openclaw-gateway node /app/openclaw.mjs channels add --channel telegram --token {TELEGRAM_BOT_TOKEN}

print("\n✅ Telegram channel added.")
print("Send the pairing code shown above to your bot on Telegram to complete setup.")

---
## Part 3: Observability with Opik

OpenClaw is now running, but we have no visibility into *how* it's making decisions — which tools it called, how long they took, what it cost.

Run `opik_observability.ipynb` to install the Opik plugin and start capturing full traces for every agent turn.

Then run `opik_evaluation.ipynb` to batch-evaluate your agent's response quality with LLM-as-a-judge metrics.